#### Tagger connection and setup

In [1]:
import TimeTagger
import itertools
import numpy as np
from TimeTagger import TimeTaggerBase, Coincidence, Coincidences, CoincidenceTimestamp, FileReader, TimeTagStream, Correlation, Counter, DelayedChannel
import time
import csv
import decimal

tagger = TimeTagger.createTimeTagger()
#tagger.setHardwareBufferSize(536870912) # was 67108864(536870912)

#frequency_channel = 1 
#PPS_channel = 2 
# Define the hardware settings here, such as trigger level or dead time. 
for ch in [1, 2, 3, 4]:
    tagger.setTriggerLevel(ch, -0.3)

tagger.setInputDelay(-1, 0) #ps
tagger.setInputDelay(-2, 0)
tagger.setInputDelay(-3, 6200)
tagger.setInputDelay(-4, 6200)

ch = [-1, -2, -3, -4]
bw = 500_000        # trial = 500 ns [ps]

In [2]:
def measure_patterns(T=30):

    T_ps = int(T * 1e12)
    N_trial = T_ps // bw

    pattern_counts = np.zeros(16, dtype=np.int64)

    last_trial = None
    last_mask = 0

    def process(data):
        nonlocal last_trial, last_mask

        ts = np.asarray(data.getTimestamps())
        cs = np.asarray(data.getChannels())

        if len(ts) == 0:
            return

        trial = (ts - data.tStart) // bw

        # -1 -> 0001, -2 -> 0010, -3 -> 0100, -4 -> 1000
        bits = 1 << (-cs - 1)

        good = (trial >= 0) & (trial < N_trial)
        trial = trial[good]
        bits = bits[good]

        if len(trial) == 0:
            return

        starts = np.r_[0, np.where(np.diff(trial) != 0)[0] + 1]
        tr = trial[starts]
        masks = np.bitwise_or.reduceat(bits, starts)

        for ti, mask in zip(tr, masks):

            if last_trial is None:
                last_trial = ti
                last_mask = mask

            elif ti == last_trial:
                last_mask |= mask

            else:
                pattern_counts[int(last_mask)] += 1
                last_trial = ti
                last_mask = mask

    stream = TimeTagger.TimeTagStream(
        tagger,
        n_max_events=1_000_000,
        channels=ch
    )

    stream.stop()
    stream.startFor(T_ps, clear=True)

    total_tags = 0

    while stream.isRunning():
        data = stream.getData()
        total_tags += data.size
        process(data)
        time.sleep(0.05)

    data = stream.getData()
    total_tags += data.size
    process(data)

    if last_trial is not None:
        pattern_counts[int(last_mask)] += 1

    # 0000
    pattern_counts[0] = N_trial - pattern_counts[1:].sum()

    return pattern_counts, N_trial, total_tags

#### Function generator setup

In [3]:
### Turn off the pulse source
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources())
# Bob
awg = rm.open_resource("USB0::0x0957::0x5707::MY53801707::INSTR")
# Alice
#awg = rm.open_resource("USB0::0x0957::0x5707::MY53802358::INSTR")
awg.timeout = 5000

awg.write("*CLS")
#awg.write("SOUR1:FUNC PULS")
#awg.write("SOUR1:FREQ 10E6")
#awg.write("SOUR1:FUNC:PULS:WIDT 5E-9")
#awg.write("SOUR1:VOLT 0.001")      # 1 Vpp
#awg.write("SOUR1:VOLT:OFFS 0") # 0 V offset
#awg.write("UNIT:ANGL SEC")
#awg.write("SOUR1:PHAS 20E-9")   # ns      # phase in ns

awg.write("SOUR2:FUNC DC")
awg.write("SOUR2:VOLT:OFFS 5.30")   # V DC
awg.write("OUTP2 ON")

print(awg.query("SYST:ERR?"))

('USB0::0x0957::0x5707::MY53801707::INSTR', 'USB0::0x0957::0x1745::SERIALxxxx::INSTR', 'ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL6::INSTR', 'USB0::0x0957::0x5707::MY53802358::0::INSTR')
-222,"Data out of range; offset;value clipped to upper limit."



In [4]:
rate = TimeTagger.Countrate(tagger, ch)
time.sleep(1)

rates = rate.getData()
for c, r in zip(ch, rates):
    print(f"CH{abs(c)}: {r:.0f} counts/s")

R_total = np.sum(rates)
mu_eff = R_total * bw * 1e-12
alpha_eff = np.sqrt(mu_eff)

print(f"Total rate = {R_total:.0f} counts/s")
print(f"mu_eff = {mu_eff:.4f}")
print(f"alpha_eff = {alpha_eff:.4f}")

CH1: 21083 counts/s
CH2: 21429 counts/s
CH3: 22773 counts/s
CH4: 19720 counts/s
Total rate = 85006 counts/s
mu_eff = 0.0425
alpha_eff = 0.2062


In [5]:
voltage_list = []
counts_list = []
Ntrial_list = []

# 1. Dark measurement
#awg.write("SOUR1:VOLT 0.001")
awg.write("SOUR2:VOLT:OFFS 5.3")
awg.write("OUTP2 ON")
time.sleep(2)
print("Signal off")
print("Start taking dark measurement...")
counts, N_trial, total_tags = measure_patterns(T=60)

voltage_list.append(5.03)
counts_list.append(counts)
Ntrial_list.append(N_trial)

print("\nDark:")
print("Total tags =", total_tags)

for mask in range(16):
    print(f"{mask:04b}: {counts[mask]}")

# 2. Coherent-state sweep

#awg.write("OUTP2 ON")
print("Start sweeping ...")

n_power = 5
voltages = np.linspace(2.35, 2.55, n_power)

for k in range(n_power):

    # set power 
    V = (voltages[k])
    awg.write(f"SOUR2:VOLT:OFFS {V}")
    #awg.write(f"SOUR1:VOLT {V}")
    print(f"Point {k+1}/{n_power}: V = {V:.2f} V")
    time.sleep(1)

    # Measure 16 click patterns
    counts, N_trial, total_tags = measure_patterns(T=60)

    voltage_list.append(V)
    counts_list.append(counts)
    Ntrial_list.append(N_trial)

    #print("Total tags =", total_tags)
    print("Pattern counts =", counts)

voltage_arr = np.array(voltage_list)
counts_arr = np.array(counts_list)
Ntrial_arr = np.array(Ntrial_list)
pattern_prob = counts_arr / Ntrial_arr[:, None]

print(awg.query("SYST:ERR?"))


Signal off
Start taking dark measurement...

Dark:
Total tags = 5095239
0000: 115003395
0001: 1208384
0010: 1244853
0011: 13065
0100: 1312724
0101: 13742
0110: 14308
0111: 151
1000: 1151290
1001: 12060
1010: 12466
1011: 140
1100: 13143
1101: 144
1110: 134
1111: 1
Start sweeping ...
Point 1/5: V = 2.35 V
Pattern counts = [28776292 12095923 12168992  5107445 13725513  5766959  5798841  2434472
 11441410  4810435  4844348  2031195  5443728  2286518  2301987   965942]
Point 2/5: V = 2.40 V
Pattern counts = [42476057 12388067 12210930  3558648 14092504  4112623  4057259  1180336
 11715914  3413829  3364413   982392  3876770  1129744  1115773   324741]
Point 3/5: V = 2.45 V
Pattern counts = [55688768 11658081 11336630  2372983 13220270  2775986  2692271   564081
 10937649  2290135  2225184   466128  2591592   541870   528249   110123]
Point 4/5: V = 2.50 V
Pattern counts = [67659732 10363059 10032824  1538304 11721657  1797733  1743175   265388
  9576714  1465915  1421372   217860  1658601  

In [9]:
print("voltage_arr =", voltage_arr)
print("counts_arr =", counts_arr)
print("Ntrial_arr =", Ntrial_arr)
print("pattern_prob =", pattern_prob)

voltage_arr = [5.03 2.35 2.4  2.45 2.5  2.55]
counts_arr = [[115003395   1208384   1244853     13065   1312724     13742     14308
        151   1151290     12060     12466       140     13143       144
        134         1]
 [ 28776292  12095923  12168992   5107445  13725513   5766959   5798841
    2434472  11441410   4810435   4844348   2031195   5443728   2286518
    2301987    965942]
 [ 42476057  12388067  12210930   3558648  14092504   4112623   4057259
    1180336  11715914   3413829   3364413    982392   3876770   1129744
    1115773    324741]
 [ 55688768  11658081  11336630   2372983  13220270   2775986   2692271
     564081  10937649   2290135   2225184    466128   2591592    541870
     528249    110123]
 [ 67659732  10363059  10032824   1538304  11721657   1797733   1743175
     265388   9576714   1465915   1421372    217860   1658601    253771
     245944     37951]
 [ 77686641   9077931   8520935    992796   9938145   1160343   1090466
     126724   8163415    953144   

In [10]:
np.savez(
    "4px_dc_sweep_data.npz",
    voltage_arr=voltage_arr,
    counts_arr=counts_arr,
    Ntrial_arr=Ntrial_arr,
    pattern_prob=pattern_prob
)

In [8]:

TimeTagger.freeTimeTagger(tagger)

#### Fitting and reconstruction